# Test New Pipeline Features

Quick validation of the refactored pipeline components:
1. **Galactic extinction** (IrsaDust)
2. **ALeRCE direct DB** (PostgreSQL)
3. **Villar SPM** peak estimates
4. **NED redshifts**
5. **End-to-end** pipeline with all features

In [1]:
import sys, os
os.chdir(os.path.join(os.path.dirname(os.path.abspath('.')), ''))
# Ensure project root is on path
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())
sys.path.insert(0, os.getcwd())

## 1. Galactic Extinction

Query IRSA dust maps for extinction at DDF coordinates.
COSMOS is at high galactic latitude so extinction should be low (~0.05-0.08 in g).

In [2]:
from utils.extinction import get_extinction, get_extinction_batch, correct_magnitude
import pandas as pd

# Test single coordinate — COSMOS DDF center
ext_cosmos = get_extinction(150.11, 2.23)
print('COSMOS extinction (A_SFD by band):')
for band, a_val in sorted(ext_cosmos.items()):
    print(f'  {band}: {a_val:.4f} mag')

print()
# Test extinction correction
raw_mag = 20.5
corrected = correct_magnitude(raw_mag, ext_cosmos.get('r', 0))
print(f'Example: raw mag = {raw_mag}, A_r = {ext_cosmos.get("r", 0):.4f}')
print(f'  Corrected mag = {corrected:.4f}')

In [3]:
# Test batch extinction on all DDF fields
from core.ddf_fields import DDF_FIELDS

ddf_df = pd.DataFrame(DDF_FIELDS)
print('DDF fields:')
display(ddf_df)

print('\nQuerying extinction for all DDFs...')
ddf_ext = get_extinction_batch(ddf_df)
print('\nExtinction values:')
display(ddf_ext[['name', 'ra', 'dec', 'A_u', 'A_g', 'A_r', 'A_i', 'A_z']])

## 2. ALeRCE Direct Database Access

Connect to ALeRCE's read-only PostgreSQL database and query SN candidates.

In [4]:
from broker_clients.alerce_db_client import AlerceDBClient

db = AlerceDBClient()
print(f'psycopg2 available: {db.available}')

if db.available:
    db.connect()
    print('Connected to ALeRCE database')
else:
    print('Install psycopg2-binary: pip install psycopg2-binary')

In [5]:
# Query SN Ia candidates — use P>0.3 threshold since ALeRCE lc_classifier
# gives SNIa lower probabilities than SLSN (no SNIa have P>0.8 as top class)
if db.available:
    candidates = db.query_sn_candidates(min_prob=0.3)
    print(f'SN candidates with P > 0.3: {len(candidates)}')
    
    # Show class distribution
    top = candidates[candidates['ranking'] == 1]
    print(f'\nTop-ranked class distribution:')
    print(top['class_name'].value_counts().to_string())
    
    # Show top SNIa
    snia = candidates[candidates['class_name'] == 'SNIa'].copy()
    snia_top = snia[snia['ranking'] == 1].sort_values('probability', ascending=False)
    print(f'\nTop 10 SNIa candidates (ranking=1):')
    display(snia_top[['oid', 'meanra', 'meandec', 'ndet', 'class_name', 'probability']].head(10))
    
    # Save a few OIDs for later tests
    test_oids = snia_top['oid'].head(5).tolist()
    print(f'\nTest OIDs: {test_oids}')

In [6]:
# Query full probabilities for test objects
if db.available and test_oids:
    probs = db.query_probabilities(test_oids)
    print(f'Probabilities for {len(probs)} objects:')
    # Show SN-related columns
    sn_cols = [c for c in probs.columns if 'SN' in c or c in ('oid', 'top_class')]
    display(probs[sn_cols])

In [7]:
# Query PS1 host galaxy data
if db.available and test_oids:
    ps1 = db.query_ps1_host(test_oids)
    print(f'PS1 host data for {len(ps1)} objects:')
    display(ps1)

## 3. Villar SPM Features

Fetch precomputed SPM parameters from ALeRCE and extract peak estimates.

In [8]:
from core.peak_fitting import villar_flux, villar_peak_from_params, extract_villar_peaks
import numpy as np

if db.available and test_oids:
    # Get SPM features
    features = db.query_features(test_oids, prefix='SPM')
    print(f'SPM features for {len(features)} objects:')
    print(f'Columns: {list(features.columns)}')
    display(features.head())
    
    # Build firstmjd lookup from candidates
    firstmjd_lookup = dict(zip(snia_top['oid'], snia_top['firstmjd']))
    
    # Extract peak estimates
    if len(features) > 0:
        villar_peaks = extract_villar_peaks(features, firstmjd_lookup)
        print(f'\nVillar peak estimates for {len(villar_peaks)} objects:')
        for oid, bands in villar_peaks.items():
            for band, result in bands.items():
                print(f'  {oid} [{band}]: peak_mjd={result["peak_mjd"]:.2f}, '
                      f'peak_mag={result["peak_mag"]:.2f}, status={result["status"]}')

In [9]:
# Plot the Villar model for one object
import matplotlib.pyplot as plt

if db.available and test_oids and len(features) > 0:
    # Pick first object with features
    test_oid = features['oid'].iloc[0]
    row = features[features['oid'] == test_oid].iloc[0]
    firstmjd = firstmjd_lookup.get(test_oid, 59000)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    for fid, band, color, ax in [(1, 'g', 'green', axes[0]), (2, 'r', 'red', axes[1])]:
        sfx = f'_{fid}'
        A = row.get(f'SPM_A{sfx}', np.nan)
        t0 = row.get(f'SPM_t0{sfx}', np.nan)
        beta = row.get(f'SPM_beta{sfx}', np.nan)
        gamma = row.get(f'SPM_gamma{sfx}', np.nan)
        tau_rise = row.get(f'SPM_tau_rise{sfx}', np.nan)
        tau_fall = row.get(f'SPM_tau_fall{sfx}', np.nan)
        
        if np.isnan(A):
            ax.text(0.5, 0.5, f'No SPM data for {band}-band', 
                    transform=ax.transAxes, ha='center')
            continue
            
        mjd_grid = np.linspace(firstmjd - 10, firstmjd + t0 + gamma + 100, 500)
        flux = villar_flux(mjd_grid, firstmjd, A, t0, beta, gamma, tau_rise, tau_fall)
        
        ax.plot(mjd_grid, flux, '-', color=color, linewidth=2)
        peak_mjd = firstmjd + t0 + gamma
        ax.axvline(peak_mjd, color='gray', linestyle='--', alpha=0.5, label=f'Peak MJD={peak_mjd:.1f}')
        ax.set_xlabel('MJD')
        ax.set_ylabel('Flux (1e26 units)')
        ax.set_title(f'{test_oid} — Villar SPM ({band}-band)')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 4. NED Redshift Lookups

Query NED for spectroscopic host galaxy redshifts near our test objects.

In [10]:
from utils.ned_query import query_ned_redshift, query_ned_batch

# Test single coordinate — well-known galaxy cluster in COSMOS field
result = query_ned_redshift(150.11, 2.23, radius_arcsec=60)
if result:
    print(f'NED result near COSMOS center:')
    print(f'  Name: {result["ned_name"]}')
    print(f'  Redshift: {result["redshift"]:.4f}')
    print(f'  Distance modulus: {result["distmod"]:.2f} mag')
    print(f'  Separation: {result["separation_arcsec"]:.1f} arcsec')
else:
    print('No NED source with redshift found near COSMOS center')

In [11]:
# Batch NED lookup for test objects
if db.available and test_oids:
    test_coords = snia_top[snia_top['oid'].isin(test_oids)][['oid', 'meanra', 'meandec']].copy()
    test_coords = test_coords.rename(columns={'meanra': 'ra', 'meandec': 'dec'})
    
    print(f'Querying NED for {len(test_coords)} objects...')
    ned_results = query_ned_batch(test_coords)
    
    print('\nNED results:')
    display(ned_results[['oid', 'ra', 'dec', 'ned_redshift', 'ned_distmod', 'ned_name', 'ned_sep_arcsec']])

## 5. Extinction + NED Combined

For objects with both extinction and redshift, compute absolute magnitudes.

In [ ]:
if db.available and test_oids:
    # Combine extinction and NED data
    combined = test_coords.copy()
    
    # Add extinction
    print('Querying extinction...')
    combined = get_extinction_batch(combined)
    
    # Add NED redshifts (already in ned_results)
    combined['ned_redshift'] = ned_results['ned_redshift'].values
    combined['ned_distmod'] = ned_results['ned_distmod'].values
    
    print('\nCombined data:')
    display(combined[['oid', 'ra', 'dec', 'A_g', 'A_r', 'ned_redshift', 'ned_distmod']])
    
    # Example: if we had a peak mag of 20.0 in r-band
    example_mag = 20.0
    for _, row in combined.iterrows():
        if pd.notna(row['A_r']) and pd.notna(row['ned_distmod']):
            m_corrected = example_mag - row['A_r']
            M_abs = m_corrected - row['ned_distmod']
            print(f"\n{row['oid']}: m_r={example_mag} -> m_corrected={m_corrected:.3f} "
                  f"-> M_abs={M_abs:.2f} (z={row['ned_redshift']:.4f})")
            if -20.5 < M_abs < -18.0:
                print('  ^ Consistent with SN Ia absolute magnitude range!')

## 6. End-to-End Pipeline Test

Run the full pipeline on one DDF field with all new features enabled.

In [ ]:
from supernova_monitor import SupernovaMonitor
from core.ddf_fields import DDF_FIELDS

# Use just COSMOS for a quick test
cosmos = [f for f in DDF_FIELDS if f['name'] == 'COSMOS']

monitor = SupernovaMonitor(
    cache_dir='./cache/data',
    use_alerce_db=True,
    apply_extinction=True,
    query_ned=True,
)

print('Pipeline initialized with all new features')
print(f'Brokers available: {list(monitor.brokers.keys())}')

In [ ]:
# Run pipeline on COSMOS only, with relaxed thresholds for testing
results = monitor.run_full_pipeline(
    min_ia_probability=0.3,
    days_back=90,
    limit=50,
    ddf_fields=cosmos,
    require_rubin=False,
    atlas_enrichment=False,  # skip ATLAS to keep test fast
)

if results is not None and len(results) > 0:
    print(f'\nPipeline returned {len(results)} candidates')
    
    # Check which new columns are present
    new_cols = [c for c in results.columns if c.startswith(('A_', 'ned_'))]
    print(f'New columns: {new_cols}')
    
    # Show a few rows with new data
    show_cols = ['unique_id', 'ra', 'dec', 'mean_ia_prob', 'num_brokers']
    show_cols += [c for c in new_cols if c in results.columns]
    display(results[show_cols].head(10))
else:
    print('No candidates returned (check broker connectivity)')

In [ ]:
# Summary statistics
if results is not None and len(results) > 0:
    print('=== New Feature Coverage ===')
    
    if 'A_g' in results.columns:
        n_ext = results['A_g'].notna().sum()
        print(f'Extinction:  {n_ext}/{len(results)} have A_g values')
        if n_ext > 0:
            print(f'  A_g range: {results["A_g"].min():.4f} - {results["A_g"].max():.4f}')
    
    if 'ned_redshift' in results.columns:
        n_z = results['ned_redshift'].notna().sum()
        print(f'NED redshift: {n_z}/{len(results)} have spectroscopic z')
        if n_z > 0:
            print(f'  z range: {results["ned_redshift"].min():.4f} - {results["ned_redshift"].max():.4f}')
    
    print(f'\nBroker breakdown:')
    print(results['brokers_detected'].value_counts().to_string())

## 7. Cache Verification

Check that extinction and NED data were cached in SQLite.

In [ ]:
from cache.alert_cache import AlertCache

cache = AlertCache('./cache/data')

# Test extinction cache retrieval (should be instant, no IRSA query)
cached_ext = cache.get_cached_extinction(150.11, 2.23)
if cached_ext:
    print('Cached extinction for COSMOS center:', cached_ext)
else:
    print('No cached extinction (run extinction test first)')

# Test NED cache
cached_ned = cache.get_cached_ned_info(150.11, 2.23)
if cached_ned:
    print('Cached NED info for COSMOS center:', cached_ned)
else:
    print('No cached NED info')